In [35]:
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm
import plotly.graph_objects as go

In [7]:
with open("../INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

In [8]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"../Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

69

In [20]:
data = pd.read_csv('000300.SH.csv', index_col = 0, parse_dates = True)

data = data[['PE_TTM', 'DIVIDENDYIELD2', 'CLOSE']]

data.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2024-11-14,12.7666,2.9248,4039.6191
2024-11-15,12.6125,2.9679,3968.8308
2024-11-18,12.6243,2.9685,3950.3806
2024-11-19,12.6549,2.9584,3976.8912
2024-11-20,12.6760,2.9522,3985.7727


In [68]:
Z_score_years = 4
short_window = 5
long_window = 10


Z_score_window = Z_score_years * 250

data['PE_Z'] = ((data['PE_TTM'] - data['PE_TTM'].rolling(Z_score_window).mean()) 
                / data['PE_TTM'].rolling(Z_score_window).std())
data['DY_Z'] = ((data['DIVIDENDYIELD2'] - data['DIVIDENDYIELD2'].rolling(Z_score_window).mean()) 
                / data['DIVIDENDYIELD2'].rolling(Z_score_window).std())

data['MAD'] = data['CLOSE'].rolling(short_window).mean() - data['CLOSE'].rolling(long_window).mean()
data['Return'] = data['CLOSE'].pct_change()

In [111]:
Z_threshold = 0
account = 10000
holding = 0
transaction = 0.0020
balance = {}

for i in tqdm(range(len(data.index))):
    date = data.index[i]
    
    if data.loc[date].isna().any():
        continue
    
    PE_Z = data.loc[date, 'PE_Z']
    DY_Z = data.loc[date, 'DY_Z']
    MAD_in = data.loc[date, 'MAD'] > 20
    MAD_out = data.loc[date, 'MAD'] < 0
    
    CLOSE = data.loc[date, 'CLOSE']
    
    balance[date] = account + holding * CLOSE / (1 + transaction)
    
    
    if PE_Z < -Z_threshold and DY_Z > Z_threshold and MAD_in and holding == 0:
        holding = account / CLOSE / (1 + transaction)
        account = 0
        
    if (PE_Z >= -Z_threshold or DY_Z <= Z_threshold or MAD_out) and holding != 0:
        account = holding * CLOSE / (1 + transaction)
        holding = 0
        
    
    
#     if PE_Z < -Z_threshold and DY_Z > Z_threshold and MAD:
#         assert holding != 0
        
#     if PE_Z > -Z_threshold or DY_Z < Z_threshold or not MAD:
#         assert holding == 0

#     if MAD and holding == 0:
#         holding = account / CLOSE / (1 + transaction)
#         account = 0
        
#     if not MAD and holding != 0:
#         account = holding * CLOSE / (1 + transaction)
#         holding = 0
        
        
    

100%|███████████████████████████████████████████████████████████████████████████| 4768/4768 [00:00<00:00, 13326.71it/s]


In [112]:
balance_df = pd.DataFrame(list(balance.items()), columns=['Date', 'Balance'])
balance_df.set_index('Date', inplace=True)

In [113]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=balance_df.index, y=balance_df['Balance'], mode='lines', name='Balance'))

# Set layout
fig.update_layout(title='Balance Over Time',
                  xaxis_title='Date',
                  yaxis_title='Balance',
                  xaxis=dict(type='date'),
                  template='plotly_white')

# Show the plot
fig.show()

In [100]:
data.index = pd.to_datetime(data.index)  # Ensure data index is datetime
balance_df.index = pd.to_datetime(balance_df.index)  # Convert balance index to datetime

# Create figure
fig = go.Figure()

# Add balance trace
fig.add_trace(go.Scatter(x=balance_df.index, y=balance_df['Balance'], mode='lines', name='Balance', yaxis='y1'))

# Add CLOSE price trace
fig.add_trace(go.Scatter(x=data.index, y=data['CLOSE'], mode='lines', name='CLOSE Price', yaxis='y2'))

# Layout with dual y-axes
fig.update_layout(
    title='Balance and Close Price Over Time',
    xaxis=dict(title='Date', type='date'),
    yaxis=dict(title='Balance', side='left', showgrid=False),
    yaxis2=dict(title='CLOSE Price', overlaying='y', side='right', showgrid=False),
    template='plotly_white'
)

# Show plot
fig.show()

In [134]:
Z_score_years = 4
short_window = 3
long_window = 15


Z_score_window = Z_score_years * 250

data['PE_Z'] = ((data['PE_TTM'] - data['PE_TTM'].rolling(Z_score_window).mean()) 
                / data['PE_TTM'].rolling(Z_score_window).std())
data['DY_Z'] = ((data['DIVIDENDYIELD2'] - data['DIVIDENDYIELD2'].rolling(Z_score_window).mean()) 
                / data['DIVIDENDYIELD2'].rolling(Z_score_window).std())

data['MAD'] = data['CLOSE'].rolling(short_window).mean() - data['CLOSE'].rolling(long_window).mean()
data['Return'] = data['CLOSE'].pct_change()




Z_threshold = 0
account = 10000
holding = 0
transaction = 0.0020
balance = {}

for i in tqdm(range(len(data.index))):
    date = data.index[i]
    
    if data.loc[date].isna().any():
        continue
        
    previous_date = data.index[i - 1]
        
    PE_in = data.loc[date, 'PE_TTM'] < 9
    PE_out = data.loc[date, 'PE_TTM'] > 12
    
    DY_in = data.loc[date, 'DIVIDENDYIELD2'] > 5
    DY_out = data.loc[date, 'DIVIDENDYIELD2'] < 3
    
    MAD_in = data.loc[date, 'MAD'] > 0 and data.loc[previous_date, 'MAD'] <= 0
    MAD_out = data.loc[date, 'MAD'] < 0 and data.loc[previous_date, 'MAD'] >= 0
    
    balance[date] = account + holding * CLOSE / (1 + transaction)
    
    if PE_in and DY_in and MAD_in and holding == 0:
        holding = account / CLOSE / (1 + transaction)
        account = 0
        
    if (PE_out or DY_out or MAD_out) and holding != 0:
        account = holding * CLOSE / (1 + transaction)
        holding = 0

100%|███████████████████████████████████████████████████████████████████████████| 4768/4768 [00:00<00:00, 12002.94it/s]


In [136]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=balance_df.index, y=balance_df['Balance'], mode='lines', name='Balance'))

# Set layout
fig.update_layout(title='Portfolio Value',
                  xaxis_title='Date',
                  yaxis_title='Balance',
                  xaxis=dict(type='date'),
                  template='plotly_white')

# Show the plot
fig.show()